In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bahushruth/legalclausedataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/bahushruth/legalclausedataset


In [19]:
import os, glob
import pandas as pd

path = "/kaggle/input/datasets/bahushruth/legalclausedataset"

for root, dirs, files in os.walk(path):
    for f in files:
        fp = os.path.join(root, f)
        print(fp, "|", round(os.path.getsize(fp)/1e6, 2), "MB")

csvs = sorted(glob.glob(path + "/**/*.csv", recursive=True))
print("\n", len(csvs), "csv files\n")
for fp in csvs[:4]:
    df = pd.read_csv(fp)
    print("FILE:", os.path.basename(fp), "| shape:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print(df.head(2).to_string()[:700])
    print("-" * 60)

/kaggle/input/datasets/bahushruth/legalclausedataset/absence-of-certain-changes-or-events.csv | 0.27 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/absence-of-certain-changes.csv | 0.27 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/acceleration.csv | 0.26 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/access-to-information.csv | 0.3 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/access.csv | 0.28 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/accounting-terms.csv | 0.21 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/additional-agreements.csv | 0.24 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/additional-documents.csv | 0.19 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/adjustments.csv | 0.25 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/affirmative-covenants.csv | 0.18 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/agreement.csv | 0.16 MB
/kaggle/input/datasets/bahushruth/legalclausedataset/a

In [20]:
import torch, torch.nn as nn

class CRF(nn.Module):
    def __init__(self, n, batch_first=True):
        super().__init__()
        self.n = n
        self.start = nn.Parameter(torch.zeros(n))
        self.end = nn.Parameter(torch.zeros(n))
        self.trans = nn.Parameter(torch.zeros(n, n))

    def _score(self, emis, tags, mask):
        mf = mask.float()
        score = self.start[tags[:, 0]] + emis[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, emis.size(1)):
            s = self.trans[tags[:, t-1], tags[:, t]] + emis[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            score = score + s * mf[:, t]
        last = mask.long().sum(1) - 1
        last_tags = tags.gather(1, last.unsqueeze(1)).squeeze(1)
        return score + self.end[last_tags]

    def _partition(self, emis, mask):
        alpha = self.start + emis[:, 0]
        for t in range(1, emis.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + self.trans.unsqueeze(0) + emis[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        return torch.logsumexp(alpha + self.end, dim=1)

    def forward(self, emis, tags, mask=None, reduction="mean"):
        ll = self._score(emis, tags, mask) - self._partition(emis, mask)
        return ll.mean() if reduction == "mean" else ll

    def decode(self, emis, mask=None):
        B, T, n = emis.shape
        score = self.start + emis[:, 0]
        hist = []
        ident = torch.arange(n, device=emis.device).unsqueeze(0).expand(B, n)
        for t in range(1, T):
            best, idx = (score.unsqueeze(2) + self.trans.unsqueeze(0)).max(1)
            nxt = best + emis[:, t]
            m = mask[:, t].unsqueeze(1)
            score = torch.where(m, nxt, score)
            hist.append(torch.where(m, idx, ident))
        cur = (score + self.end).argmax(1)
        path = [cur]
        for k in range(T - 2, -1, -1):
            cur = hist[k].gather(1, cur.unsqueeze(1)).squeeze(1)
            path.append(cur)
        path = torch.stack(path[::-1], dim=1)
        lens = mask.long().sum(1).tolist()
        return [path[b, :lens[b]].tolist() for b in range(B)]

def get_entities(tags):
    ents, cur = set(), None
    for i, t in enumerate(list(tags) + ["O"]):
        if cur and (t == "O" or t.startswith("B-") or t[2:] != cur[0]):
            ents.add((cur[0], cur[1], i)); cur = None
        if t.startswith("B-") or (t.startswith("I-") and cur is None):
            cur = (t[2:], i)
    return ents

def ner_scores(y_true, y_pred):
    """Entity-level precision/recall/F1 per label, plus overall."""
    stats = {}
    for a, b in zip(y_true, y_pred):
        ea, eb = get_entities(a), get_entities(b)
        for lab, s, e in eb | ea:
            d = stats.setdefault(lab, [0, 0, 0])      # tp, fp, fn
            if (lab, s, e) in ea and (lab, s, e) in eb: d[0] += 1
            elif (lab, s, e) in eb: d[1] += 1
            else: d[2] += 1
    stats["ALL"] = [sum(v[i] for k, v in stats.items()) for i in range(3)]
    print(f"{'label':8} {'precision':>9} {'recall':>8} {'f1':>8}")
    for lab, (tp, fp, fn) in stats.items():
        p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
        print(f"{lab:8} {p:9.3f} {r:8.3f} {2*p*r/max(p+r, 1e-9):8.3f}")
    return stats["ALL"][0] / max(stats["ALL"][0] + stats["ALL"][1], 1)

In [21]:
import importlib
for m in ["torch", "tokenizers", "gensim", "sklearn"]:
    try:
        importlib.import_module(m); print(m, "OK")
    except Exception:
        print(m, "MISSING")

torch OK
tokenizers OK
gensim OK
sklearn OK


In [22]:
def load_all(csvs):
    frames = []
    for fp in csvs:
        df = pd.read_csv(fp)
        df.columns = [str(c).strip().lower() for c in df.columns]
        obj = [c for c in df.columns if df[c].dtype == object]
        text_col = max(obj, key=lambda c: df[c].astype(str).str.len().mean())
        lab_cols = [c for c in obj if c != text_col and df[c].nunique() < 200]
        out = pd.DataFrame({"text": df[text_col].astype(str)})
        out["label"] = (df[lab_cols[0]].astype(str) if lab_cols
                        else os.path.splitext(os.path.basename(fp))[0])
        frames.append(out)
    return pd.concat(frames, ignore_index=True)

data_df = load_all(csvs)
data_df = data_df.dropna().drop_duplicates(subset="text").reset_index(drop=True)
data_df["words"] = data_df["text"].str.split().str.len()

print("rows:", len(data_df))
print("\nLabel counts:")
print(data_df["label"].value_counts().to_string())
print("\nWords per clause:")
print(data_df["words"].describe().round(1).to_string())
print("\nSample:")
print(data_df.sample(3, random_state=0)[["label", "text"]].to_string()[:1200])

rows: 150545

Label counts:
label
time-of-essence                                  630
time-of-the-essence                              620
definitions-and-interpretation                   590
capitalized-terms                                590
captions                                         580
now-therefore                                    570
section-headings                                 570
exhibits                                         570
headings                                         570
financial-covenants                              560
defined-terms                                    560
definitions                                      560
certain-defined-terms                            550
covenants-of-the-company                         540
name                                             540
currency                                         539
effective-date                                   530
conditions-to-effectiveness                      530
special-term

In [ ]:
import re, json, random, os, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from gensim.models import Word2Vec
from sklearn.metrics import classification_report as cls_report
from sklearn.model_selection import train_test_split
print("imports OK | torch", torch.__version__)

In [ ]:
path = "/kaggle/input/datasets/bahushruth/legalclausedataset"

# ---- load (skipped if data_df already exists in this session) ----
try:
    data_df
except NameError:
    csvs = sorted(glob.glob(path + "/**/*.csv", recursive=True))
    frames = []
    for fp in csvs:
        d = pd.read_csv(fp)
        d.columns = [str(c).strip().lower() for c in d.columns]
        obj = [c for c in d.columns if d[c].dtype == object]
        tcol = max(obj, key=lambda c: d[c].astype(str).str.len().mean())
        frames.append(pd.DataFrame({"text": d[tcol].astype(str),
                                    "label": os.path.splitext(os.path.basename(fp))[0]}))
    data_df = pd.concat(frames, ignore_index=True)

RISK = {
    "Indemnification": ["indemnification", "indemnity", "indemnification-by-the-company",
                        "indemnification-and-contribution"],
    "Limitation of Liability": ["limitation-of-liability", "limitation-on-liability"],
    "Force Majeure": ["force-majeure"],
}
OTHER = ["governing-law", "confidentiality", "termination", "notices", "assignment", "severability",
         "entire-agreement", "amendment", "waiver", "counterparts", "payment-terms", "insurance",
         "survival", "remedies", "warranty", "set-off", "release", "guarantee",
         "representations-and-warranties", "attorneys-fees", "arbitration", "non-solicitation",
         "publicity", "taxes", "intellectual-property", "interest", "fees", "dispute-resolution",
         "jurisdiction", "further-assurances", "no-waiver", "successors-and-assigns",
         "relationship-of-parties", "injunctive-relief", "specific-performance", "exclusivity"]
lab2cls = {l: c for c, ls in RISK.items() for l in ls}
lab2cls.update({l: "Other" for l in OTHER})

STOPW = {"of", "and", "the", "on", "for", "to", "in", "by", "a"}
def norm(t): return re.sub(r"\s+", " ", str(t)).strip()

def strip_hint(text, label):
    """Remove a leading heading like '1.1 Indemnification.' that gives away the label."""
    lab = {w for w in re.split(r"[-_]", label) if w not in STOPW}
    changed = False
    for _ in range(3):
        m = re.match(r"^\W{0,3}(.{1,80}?)[\.:]\s+(?=\S)", text)
        if not m: break
        head = m.group(1).strip(); low = head.lower()
        numbering = re.fullmatch(r"(section|article|sec|clause)?\s*[\divxa-z\.\(\)]{1,8}", low) is not None
        if len(head.split()) <= 8 and (numbering or any(w in low for w in lab)):
            text = text[m.end():]; changed = True
        else:
            break
    return text, changed

df = data_df[data_df["label"].isin(lab2cls)].copy()
df["orig"] = df["text"].map(norm)
pairs = [strip_hint(t, l) for t, l in zip(df["orig"], df["label"])]
df["text"] = [p[0] for p in pairs]
df["stripped"] = [p[1] for p in pairs]
df["words"] = df["text"].str.split().str.len()
df = df[(df.words >= 8) & (df.words <= 120)]
df = df.drop_duplicates(subset="text")
df["cls"] = df["label"].map(lab2cls)

parts = []
for (lab, cls), g in df.groupby(["label", "cls"]):
    parts.append(g.sample(min(len(g), 50), random_state=0) if cls == "Other" else g)
clauses = pd.concat(parts).reset_index(drop=True)

print("Class counts:\n", clauses["cls"].value_counts().to_string())
print("\n% of clauses where a heading was stripped:\n",
      (clauses.groupby("cls")["stripped"].mean() * 100).round(1).to_string())
print("\nWords per clause:", clauses["words"].describe().round(1)[["mean", "50%", "max"]].to_dict())
for c in ["Indemnification", "Limitation of Liability", "Force Majeure", "Other"]:
    print(f"\n--- {c} ---")
    for _, row in clauses[clauses.cls == c].sample(2, random_state=1).iterrows():
        print("BEFORE:", row["orig"][:140]); print("AFTER :", row["text"][:140]); print()

In [23]:
MONTHS = ("January|February|March|April|May|June|July|August|September|October|November|December|"
          r"Jan\.?|Feb\.?|Mar\.?|Apr\.?|Jun\.?|Jul\.?|Aug\.?|Sept?\.?|Oct\.?|Nov\.?|Dec\.?")
DATE_RE = re.compile(
    rf"\b(?:{MONTHS})\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b"
    rf"|\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:day\s+of\s+)?(?:{MONTHS}),?\s+\d{{4}}\b"
    r"|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"
    rf"|\b(?:{MONTHS})\s+\d{{4}}\b")
MONEY_RE = re.compile(
    r"(?<![A-Za-z])(?:US\$|U\.S\.\s?\$|\$|USD|EUR|GBP|INR|Rs\.?|€|£)\s?\d(?:[\d,]*\d)?(?:\.\d+)?"
    r"(?:\s(?:million|billion|thousand|crore|lakh))?")
SUFFIX = r"(?:Inc\.?|Incorporated|LLC|L\.L\.C\.|Ltd\.?|Limited|Corp\.?|Corporation|Company|Co\.|LLP|L\.P\.|LP|PLC|plc|GmbH|N\.A\.)"
ORG_RE = re.compile(rf"(?:[A-Z][\w&'\-]*,?\s+(?:(?:of|and|&|the)\s+)?){{1,6}}?{SUFFIX}(?=\s|,|;|\)|\.|$)")
STOP = {"The","This","That","By","And","Between","Among","With","From","To","Whereas","Now","Each","Any",
        "If","In","Such","Said","Of","For","On","At","Or","As","Is","Are","Its","Their","An","A","No",
        "All","Both","Either","Neither","Except","Upon","Under","Parent","Buyer","Seller"}

def trim_org(text, s, e):
    span = text[s:e]
    while True:
        m = re.match(r"([A-Za-z]+)\s+", span)
        if m and m.group(1) in STOP and len(span) > m.end():
            s += m.end(); span = text[s:e]
        else:
            break
    return s, e

def silver_spans(text):
    cands = []
    for lab, rx in [("MONEY", MONEY_RE), ("DATE", DATE_RE), ("ORG", ORG_RE)]:
        for m in rx.finditer(text):
            s, e = m.start(), m.end()
            if lab == "ORG":
                s, e = trim_org(text, s, e)
                if " " not in text[s:e].strip(): continue
            cands.append((s, e, lab))
    cands.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    out, last = [], -1
    for s, e, lab in cands:
        if s >= last: out.append((s, e, lab)); last = e
    return out

def show(text, spans):
    out, pos = "", 0
    for s, e, lab in spans:
        out += text[pos:s] + f"[{text[s:e]}|{lab}]"; pos = e
    return out + text[pos:]

counts = {"ORG": 0, "DATE": 0, "MONEY": 0}; with_ent = 0
for t in clauses["text"]:
    sp = silver_spans(t); with_ent += bool(sp)
    for _, _, lab in sp: counts[lab] += 1
print("silver entity counts:", counts, "| clauses with >=1 entity:", with_ent, "of", len(clauses))

ex = [t for t in clauses["text"].sample(300, random_state=0) if silver_spans(t)][:6]
for t in ex:
    print(show(t, silver_spans(t))[:350]); print()

silver entity counts: {'ORG': 69, 'DATE': 7, 'MONEY': 21} | clauses with >=1 entity: 81 of 2570
Except as otherwise set forth in the Registration Statement, the Prospectus or the [Time of Sale Prospectus, the Company|ORG] is insured by insurers of recognized financial responsibility against such losses and risks and in such amounts as are adequate for the conduct of its businesses and as are customary for the business in which it is engaged; 

(a) On the Effective Date, the Fund shall pay up to [$25,000|MONEY] of the fees and expenses of Banc of America's outside counsel in connection with the negotiation and documentation of the transactions contemplated by this Agreement and the initial organization and set up of a voting trust to be formed with respect to the AMTP Shares (the "Voting 

During the period beginning the date hereof and ending on the first to occur of (a) the Closing and (b) the termination of this Agreement pursuant to Section 9.1, neither [Sellers, the Company|ORG], n

In [24]:
CONF = {"l":"1","I":"l","O":"0","0":"O","m":"rn","e":"c","a":"o","s":"5","B":"8"}
def noise(s, p, r):
    if p <= 0: return s
    out = []
    for ch in s:
        x = r.random()
        if x < p*0.35 and ch in CONF: out.append(CONF[ch])
        elif x < p*0.55 and ch in ",.;:": pass
        elif x < p*0.70 and ch.isalpha(): pass
        elif x < p*0.75 and ch == " ": pass
        else: out.append(ch)
    return "".join(out)

def noisy_example(text, spans, p, r):
    """Apply OCR noise while keeping entity spans aligned to the noisy text."""
    if p <= 0: return text, list(spans)
    out, new, pos = "", [], 0
    for s, e, lab in sorted(spans):
        out += noise(text[pos:s], p, r)
        ent = noise(text[s:e], p/2, r)
        if ent.strip():
            new.append((len(out), len(out) + len(ent), lab)); out += ent
        pos = e
    out += noise(text[pos:], p, r)
    return out, new

tr, tmp = train_test_split(clauses, test_size=0.3, stratify=clauses["cls"], random_state=0)
va, te = train_test_split(tmp, test_size=0.5, stratify=tmp["cls"], random_state=0)

def build(frame, levels, seed, copies=1):
    r = random.Random(seed); out = []
    for text, cls in zip(frame["text"], frame["cls"]):
        spans = silver_spans(text)                # labelled on CLEAN text
        for _ in range(copies):
            p = r.choice(levels)
            t, sp = noisy_example(text, spans, p, r)
            if t.strip():
                out.append({"text": t, "spans": sp, "clause": cls, "noise": p})
    return out

train_set = build(tr, (0, 0.02, 0.05, 0.1), 1, copies=2)
val_set   = build(va, (0.05,), 2)
test_sets = {"clean": build(te, (0,), 3), "light OCR": build(te, (0.03,), 4), "heavy OCR": build(te, (0.1,), 5)}

print("train:", len(train_set), "| val:", len(val_set), "| test each:", len(test_sets["clean"]))
d = next(x for x in train_set if x["noise"] >= 0.05 and x["spans"])
print("\nNOISY EXAMPLE:\n", show(d["text"], d["spans"])[:400])

train: 3598 | val: 385 | test each: 386

NOISY EXAMPLE:
 Interest shal be payabl monthly, in arrears, beginning [pil 1, 2010|DATE] nd ontinuing on the first day ofcach oseutive month threafter until [September 30, 2010|DATE] (“aturity Dae”), whenone final paymento the entirebalance of principal, interes, es, premis, charges and costs and expen5es thenoustanding  this Note shall b due and ayble in full.


In [25]:
MONTHS = ("January|February|March|April|May|June|July|August|September|October|November|December|"
          r"Jan\.?|Feb\.?|Mar\.?|Apr\.?|Jun\.?|Jul\.?|Aug\.?|Sept?\.?|Oct\.?|Nov\.?|Dec\.?")
DATE_RE = re.compile(
    rf"\b(?:{MONTHS})\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b"
    rf"|\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:day\s+of\s+)?(?:{MONTHS}),?\s+\d{{4}}\b"
    r"|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"
    rf"|\b(?:{MONTHS})\s+\d{{4}}\b")
MONEY_RE = re.compile(
    r"(?<![A-Za-z])(?:US\$|U\.S\.\s?\$|\$|USD|EUR|GBP|INR|Rs\.?|€|£)\s?\d(?:[\d,]*\d)?(?:\.\d+)?"
    r"(?:\s(?:million|billion|thousand|crore|lakh))?")
SUFFIX = (r"(?:Pvt\.\s+Ltd\.?|Private\s+Limited|Inc\.?|Incorporated|LLC|L\.L\.C\.|Ltd\.?|Limited|"
          r"Corp\.?|Corporation|Company|Co\.|LLP|L\.P\.|LP|PLC|plc|GmbH|N\.A\.)")
W = r"[A-Z][\w&'\-]*"
ORG_RE = re.compile(rf"{W}(?:\s+(?:(?:of|and|&)\s+)?{W}){{0,5}}?,?\s+{SUFFIX}(?=\s|,|;|\)|\.|$)")
STOP = {"The","This","That","By","And","Between","Among","With","From","To","Whereas","Now","Each","Any",
        "If","In","Such","Said","Of","For","On","At","Or","As","Is","Are","Its","Their","An","A","No",
        "All","Both","Either","Neither","Except","Upon","Under","Parent","Buyer","Seller"}

def trim_org(text, s, e):
    span = text[s:e]
    while True:
        m = re.match(r"([A-Za-z]+)\s+", span)
        if m and m.group(1) in STOP and len(span) > m.end():
            s += m.end(); span = text[s:e]
        else:
            break
    return s, e

def silver_spans(text):
    cands = []
    for lab, rx in [("MONEY", MONEY_RE), ("DATE", DATE_RE), ("ORG", ORG_RE)]:
        for m in rx.finditer(text):
            s, e = m.start(), m.end()
            if lab == "ORG":
                s, e = trim_org(text, s, e)
                if " " not in text[s:e].strip(): continue
            cands.append((s, e, lab))
    cands.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    out, last = [], -1
    for s, e, lab in cands:
        if s >= last: out.append((s, e, lab)); last = e
    return out

def show(text, spans):
    out, pos = "", 0
    for s, e, lab in spans:
        out += text[pos:s] + f"[{text[s:e]}|{lab}]"; pos = e
    return out + text[pos:]

counts = {"ORG": 0, "DATE": 0, "MONEY": 0}; with_ent = 0
for t in clauses["text"]:
    sp = silver_spans(t); with_ent += bool(sp)
    for _, _, lab in sp: counts[lab] += 1
print("natural entities:", counts, "| clauses with >=1:", with_ent, "of", len(clauses))
for t in [t for t in clauses["text"].sample(400, random_state=0) if any(l == "ORG" for _, _, l in silver_spans(t))][:4]:
    print(show(t, silver_spans(t))[:300]); print()

natural entities: {'ORG': 45, 'DATE': 7, 'MONEY': 21} | clauses with >=1: 60 of 2570
It is understood that the Attorney-in-Fact assumes no responsibility or liability to any Person by virtue of the Power of Attorney granted by the [Management Company|ORG] hereby. The Attorney-in-Fact makes no representations with respect to and shall have no responsibility in its capacity as Attorne

Pursuant to ss.3803(a) of the Delaware Business Trust Act, 12 Del. C. ss.3801, et seq., the Holders of the STRYPES shall be entitled to the same limitation of personal liability extended to stockholders of private corporations for profit organized under the [General Corporation|ORG] Law of the State

1.1 Subject to the exceptions in Clause 6 of these Terms and Conditions, all Content included on the Website, unless uploaded by Users, including, but not limited to, text, graphics, logos, icons, images, sound clips, video clips, data compilations, page layout, underlying code and software is the p

The Stock

In [26]:
CONF = {"l":"1","I":"l","O":"0","0":"O","m":"rn","e":"c","a":"o","s":"5","B":"8"}
def noise(s, p, r):
    if p <= 0: return s
    out = []
    for ch in s:
        x = r.random()
        if x < p*0.35 and ch in CONF: out.append(CONF[ch])
        elif x < p*0.55 and ch in ",.;:": pass
        elif x < p*0.70 and ch.isalpha(): pass
        elif x < p*0.75 and ch == " ": pass
        else: out.append(ch)
    return "".join(out)

def noisy_example(text, spans, p, r):
    if p <= 0: return text, list(spans)
    out, new, pos = "", [], 0
    for s, e, lab in sorted(spans):
        out += noise(text[pos:s], p, r)
        ent = noise(text[s:e], p/2, r)
        if ent.strip():
            new.append((len(out), len(out) + len(ent), lab)); out += ent
        pos = e
    out += noise(text[pos:], p, r)
    return out, new

# ---------- entity generators (train and test use DIFFERENT name pools) ----------
POOLS = {
 "train": (["Blue","Apex","Northern","Silverline","Orion","Pioneer","Zenith","Crestview","Globex","Initech",
            "Umbrella","Hooli","Summit","Falcon","Cedar","Lunar","Maple","Nimbus","Quartz","Raven"],
           ["Harbor","Dynamics","Systems","Logistics","Energy","Software","Capital","Holdings","Foods",
            "Networks","Labs","Media","Motors","Pharma"]),
 "test":  (["Acme","Vertex","Meridian","Quantum","Lakeshore","Redwood","Bharat","Sunrise"],
           ["Partners","Industries","Solutions","Ventures","Group","Technologies"]),
}
SUFFIXES = ["Inc.","LLC","Ltd.","Pvt. Ltd.","Corp.","LLP","Limited","GmbH","Corporation"]
MONTH_LIST = ["January","February","March","April","May","June","July","August","September",
              "October","November","December"]

def gen_org(r, pool):
    first, second = POOLS[pool]
    name = r.choice(first) if r.random() < 0.3 else f"{r.choice(first)} {r.choice(second)}"
    return f"{name} {r.choice(SUFFIXES)}"
def gen_date(r):
    d, m, y = r.randint(1, 28), r.randint(1, 12), r.randint(2012, 2030)
    return r.choice([f"{MONTH_LIST[m-1]} {d}, {y}", f"{d} {MONTH_LIST[m-1]} {y}",
                     f"{d:02d}/{m:02d}/{y}", f"{d}th day of {MONTH_LIST[m-1]}, {y}"])
def gen_money(r):
    n = r.choice([5000, 25000, 75000, 250000, 1000000, 2500000])
    return r.choice([f"${n:,}", f"USD {n:,}", f"Rs. {n:,}", f"INR {n:,}", f"EUR {n:,}"])

DEFINED = re.compile(r"\b(?:[Tt]he\s+)?(Company|Seller|Buyer|Lender|Borrower|Tenant|Landlord|Contractor|"
                     r"Purchaser|Licensor|Licensee|Supplier|Client|Employer|Fund|Agent)\b")
TAILS = [" This clause shall take effect on {DATE}.",
         " Payments under this clause shall be made in the amount of {MONEY}.",
         " Notices shall be delivered to {ORG} on or before {DATE}.",
         " {ORG} shall pay a fee of {MONEY}.",
         " The parties acknowledge the position as at {DATE}.",
         " Any such notice shall be sent to {ORG}."]

def fill_tail(t, r, pool):
    out, spans, pos = "", [], 0
    for m in re.finditer(r"\{(ORG|DATE|MONEY)\}", t):
        out += t[pos:m.start()]
        val = {"ORG": lambda: gen_org(r, pool), "DATE": lambda: gen_date(r),
               "MONEY": lambda: gen_money(r)}[m.group(1)]()
        spans.append((len(out), len(out) + len(val), m.group(1))); out += val; pos = m.end()
    return out + t[pos:], spans

def inject(text, r, pool, p_org=0.6, p_tail=0.7):
    names, segs, pos = {}, [], 0
    for m in DEFINED.finditer(text):
        term = m.group(1)
        if term not in names:
            names[term] = gen_org(r, pool) if r.random() < p_org else None
        if names[term]:
            segs.append((text[pos:m.start()], None)); segs.append((names[term], "ORG")); pos = m.end()
    segs.append((text[pos:], None))
    new, spans = "", []
    for s, lab in segs:
        if lab: spans.append((len(new), len(new) + len(s), lab))
        new += s
    if r.random() < p_tail:
        for _ in range(r.choice([1, 1, 2])):
            tail_text, tsp = fill_tail(r.choice(TAILS), r, pool)
            bounds = [m.start() for m in re.finditer(r"(?<=[\.;])\s+(?=[A-Z])", new)] + [len(new)]
            b = r.choice(bounds); tail = " " + tail_text
            new = new[:b] + tail + new[b:]
            spans = [(s + len(tail), e + len(tail), l) if s >= b else (s, e, l) for s, e, l in spans]
            spans += [(s + b + 1, e + b + 1, l) for s, e, l in tsp]
    for s, e, l in silver_spans(new):              # add natural regex entities that don't overlap
        if not any(s < e2 and e > s2 for s2, e2, _ in spans): spans.append((s, e, l))
    return new, sorted(spans)

def make_base(frame, seed, pool, copies=1):
    r = random.Random(seed); out = []
    for text, cls in zip(frame["text"], frame["cls"]):
        for _ in range(copies):
            t, sp = inject(text, r, pool)
            out.append({"text": t, "spans": sp, "clause": cls})
    return out

def noisify(base, p, seed):
    r = random.Random(seed); out = []
    for d in base:
        t, sp = noisy_example(d["text"], d["spans"], p, r)
        if t.strip(): out.append({"text": t, "spans": sp, "clause": d["clause"], "noise": p})
    return out

tr, tmp = train_test_split(clauses, test_size=0.3, stratify=clauses["cls"], random_state=0)
va, te = train_test_split(tmp, test_size=0.5, stratify=tmp["cls"], random_state=0)

rr = random.Random(11); train_set = []
for d in make_base(tr, 1, "train", copies=2):
    train_set += noisify([d], rr.choice((0, 0.02, 0.05, 0.1)), rr.randint(0, 10**6))
val_set = noisify(make_base(va, 2, "train"), 0.05, 21)
test_base = make_base(te, 3, "test")                      # UNSEEN company names
test_sets = {"clean": noisify(test_base, 0, 31), "light OCR": noisify(test_base, 0.03, 32),
             "heavy OCR": noisify(test_base, 0.1, 33)}

ec = {"ORG": 0, "DATE": 0, "MONEY": 0}
for d in train_set:
    for _, _, l in d["spans"]: ec[l] += 1
print("train:", len(train_set), "| val:", len(val_set), "| test each:", len(test_sets["clean"]))
print("entities in train:", ec)
print("test clauses per class:", pd.Series([d["clause"] for d in test_sets["clean"]]).value_counts().to_dict())
for d in [x for x in train_set if x["noise"] == 0][:2] + [x for x in train_set if x["noise"] >= 0.05][:1]:
    print("\n", show(d["text"], d["spans"])[:420])

train: 3598 | val: 385 | test each: 386
entities in train: {'ORG': 3807, 'DATE': 1674, 'MONEY': 1168}
test clauses per class: {'Other': 269, 'Limitation of Liability': 58, 'Indemnification': 33, 'Force Majeure': 26}

 The covenants, agreements, representations and warranties contained in or made in Sections 4, 5, 7 and 9 of this Agreement shall survive any termination of Executive’s employment.  [Hooli Foods LLP|ORG] shall pay a fee of [INR 5,000|MONEY].  [Maple Capital Inc.|ORG] shall pay a fee of [USD 250,000|MONEY].

 For purposes of Section 14 of the Distribution Agreement, the Trust’s communications details are as set forth in Section E of the Omnibus Instrument.

 The Issuer will pay theInenture Trustee as compen5ation for performing is obligtions under this Indenture a fee separatly areed by the Issuer and the Indenture Trustee. The Indenture Trustee’s compesain will not be limited y law on compnsation ofa trusteeof an express trut.  Notices shall bedelivered to [Cedar GmbH|ORG]

In [27]:
ENT = ["ORG", "DATE", "MONEY"]
TAGS = ["O"] + [f"{p}-{e}" for e in ENT for p in "BI"]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
CLASSES = ["Indemnification", "Limitation of Liability", "Force Majeure", "Other"]
RISKS = CLASSES[:3]
CLS2ID = {c: i for i, c in enumerate(CLASSES)}

def train_tokenizer(texts, vocab_size=8000):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tr_ = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]"])
    tok.train_from_iterator(texts, tr_)
    return tok

def train_w2v(tok, texts, dim=100):
    sents = [tok.encode(t).tokens for t in texts]
    w2v = Word2Vec(sents, vector_size=dim, window=5, min_count=1, sg=1, epochs=15, seed=0, workers=1)
    mat = np.random.normal(0, 0.1, (tok.get_vocab_size(), dim)).astype("float32"); mat[0] = 0
    for t, i in tok.get_vocab().items():
        if t in w2v.wv: mat[i] = w2v.wv[t]
    return mat

def encode(tok, text, spans=None):
    enc = tok.encode(text); tags, prev = [], None
    for a, b in enc.offsets:
        hit = None
        for k, (s, e, lab) in enumerate(spans or []):
            if a < e and b > s: hit = k; break
        if hit is None: tags.append("O"); prev = None
        else: tags.append(("I-" if prev == hit else "B-") + spans[hit][2]); prev = hit
    return enc.ids, enc.offsets, tags

random.seed(0); torch.manual_seed(0); np.random.seed(0)
texts = [d["text"] for d in train_set]
tok = train_tokenizer(texts); tok.save("/kaggle/working/tok.json")
emb = train_w2v(tok, texts)

def to_data(ds):
    out = []
    for d in ds:
        ids, _, tags = encode(tok, d["text"], d["spans"])
        if ids: out.append((ids, tags, CLS2ID[d["clause"]]))
    return out
train_data, val_data = to_data(train_set), to_data(val_set)
lens = [len(x[0]) for x in train_data]
print("vocab:", tok.get_vocab_size(), "| emb:", emb.shape, "| samples:", len(train_data),
      "| tokens mean/max:", int(np.mean(lens)), max(lens))




vocab: 8000 | emb: (8000, 100) | samples: 3598 | tokens mean/max: 95 237


In [28]:
class CRF(nn.Module):
    def __init__(self, n, batch_first=True):
        super().__init__()
        self.n = n
        self.start = nn.Parameter(torch.zeros(n)); self.end = nn.Parameter(torch.zeros(n))
        self.trans = nn.Parameter(torch.zeros(n, n))

    def _score(self, emis, tags, mask):
        mf = mask.float()
        e = emis.gather(2, tags.unsqueeze(2)).squeeze(2)
        tr_ = self.trans[tags[:, :-1], tags[:, 1:]]
        score = self.start[tags[:, 0]] + (e * mf).sum(1) + (tr_ * mf[:, 1:]).sum(1)
        last = mask.long().sum(1) - 1
        return score + self.end[tags.gather(1, last.unsqueeze(1)).squeeze(1)]

    def _partition(self, emis, mask):
        alpha = self.start + emis[:, 0]
        for t in range(1, emis.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + self.trans.unsqueeze(0) + emis[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        return torch.logsumexp(alpha + self.end, dim=1)

    def forward(self, emis, tags, mask=None, reduction="mean"):
        ll = self._score(emis, tags, mask) - self._partition(emis, mask)
        return ll.mean() if reduction == "mean" else ll

    def decode(self, emis, mask=None):
        B, T, n = emis.shape
        score = self.start + emis[:, 0]; hist = []
        ident = torch.arange(n).unsqueeze(0).expand(B, n)
        for t in range(1, T):
            best, idx = (score.unsqueeze(2) + self.trans.unsqueeze(0)).max(1)
            m = mask[:, t].unsqueeze(1)
            score = torch.where(m, best + emis[:, t], score)
            hist.append(torch.where(m, idx, ident))
        cur = (score + self.end).argmax(1); path = [cur]
        for k in range(T - 2, -1, -1):
            cur = hist[k].gather(1, cur.unsqueeze(1)).squeeze(1); path.append(cur)
        path = torch.stack(path[::-1], dim=1)
        lens_ = mask.long().sum(1).tolist()
        return [path[b, :lens_[b]].tolist() for b in range(B)]

def get_entities(tags):
    ents, cur = set(), None
    for i, t in enumerate(list(tags) + ["O"]):
        if cur and (t == "O" or t.startswith("B-") or t[2:] != cur[0]):
            ents.add((cur[0], cur[1], i)); cur = None
        if t.startswith("B-") or (t.startswith("I-") and cur is None):
            cur = (t[2:], i)
    return ents

def ner_scores(y_true, y_pred):
    stats = {}
    for a, b in zip(y_true, y_pred):
        ea, eb = get_entities(a), get_entities(b)
        for lab, s, e in eb | ea:
            d = stats.setdefault(lab, [0, 0, 0])
            if (lab, s, e) in ea and (lab, s, e) in eb: d[0] += 1
            elif (lab, s, e) in eb: d[1] += 1
            else: d[2] += 1
    stats["ALL"] = [sum(v[i] for k, v in stats.items()) for i in range(3)]
    print(f"{'label':8} {'precision':>9} {'recall':>8} {'f1':>8}")
    for lab, (tp, fp, fn) in stats.items():
        p = tp / max(tp + fp, 1); r_ = tp / max(tp + fn, 1)
        print(f"{lab:8} {p:9.3f} {r_:8.3f} {2*p*r_/max(p+r_, 1e-9):8.3f}")
    return stats["ALL"][0] / max(stats["ALL"][0] + stats["ALL"][1], 1)

class Model(nn.Module):
    def __init__(self, emb, n_tags, n_cls, cls_w=None, hidden=128):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb, dtype=torch.float), freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(emb.shape[1], hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.tag_fc = nn.Linear(2*hidden, n_tags)
        self.crf = CRF(n_tags, batch_first=True)
        self.cls_fc = nn.Linear(2*hidden, n_cls)
        self.register_buffer("cls_w", cls_w if cls_w is not None else torch.ones(n_cls))

    def encode(self, x):
        h, _ = self.lstm(self.drop(self.emb(x))); return self.drop(h)
    def pool(self, h, m):
        mf = m.unsqueeze(-1).float(); return (h * mf).sum(1) / mf.sum(1)
    def cls_logits(self, x, m):
        return self.cls_fc(self.pool(self.encode(x), m))
    def loss(self, x, m, tags, y):
        h = self.encode(x)
        ner = -self.crf(self.tag_fc(h), tags, mask=m, reduction="mean")
        cls = F.cross_entropy(self.cls_fc(self.pool(h, m)), y, weight=self.cls_w)
        return ner + cls
    def predict(self, x, m):
        h = self.encode(x)
        return self.crf.decode(self.tag_fc(h), mask=m), self.cls_fc(self.pool(h, m))

def batchify(items):
    L = max(len(i[0]) for i in items)
    x = torch.zeros(len(items), L, dtype=torch.long); y = torch.zeros(len(items), L, dtype=torch.long)
    m = torch.zeros(len(items), L, dtype=torch.bool)
    for k, (ids, tags, c) in enumerate(items):
        n = len(ids); x[k, :n] = torch.tensor(ids)
        y[k, :n] = torch.tensor([TAG2ID[t] for t in tags]); m[k, :n] = True
    return x, y, m, torch.tensor([i[2] for i in items])

cnt = np.bincount([c for _, _, c in train_data], minlength=len(CLASSES))
cls_w = torch.tensor((cnt.sum() / (len(cnt) * cnt)) ** 0.5, dtype=torch.float)
print("class counts:", dict(zip(CLASSES, cnt.tolist())), "| weights:", cls_w.numpy().round(2))
model = Model(emb, len(TAGS), len(CLASSES), cls_w)
x, y, m, c = batchify(train_data[:8])
print("initial loss:", model.loss(x, m, y, c).item())

class counts: {'Indemnification': 300, 'Limitation of Liability': 534, 'Force Majeure': 250, 'Other': 2514} | weights: [1.73 1.3  1.9  0.6 ]
initial loss: 204.37567138671875


In [29]:
import time
def make_batches(data, bs=32):
    idx = list(range(len(data))); random.shuffle(idx)
    chunks = [idx[i:i+bs*20] for i in range(0, len(idx), bs*20)]; out = []
    for ch in chunks:
        ch.sort(key=lambda i: len(data[i][0])); out += [ch[i:i+bs] for i in range(0, len(ch), bs)]
    random.shuffle(out); return out

def val_accuracy():
    model.eval(); ok = n = 0
    with torch.no_grad():
        for i in range(0, len(val_data), 64):
            x, y, m, c = batchify(val_data[i:i+64])
            ok += (model.cls_logits(x, m).argmax(1) == c).sum().item(); n += len(c)
    model.train(); return ok / n

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for ep in range(8):
    t0 = time.time(); model.train(); tot = 0; nb = 0
    for b in make_batches(train_data):
        x, y, m, c = batchify([train_data[i] for i in b])
        loss = model.loss(x, m, y, c)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        tot += loss.item(); nb += 1
    print(f"epoch {ep+1}  loss {tot/nb:.4f}  val clause acc {val_accuracy():.3f}  ({time.time()-t0:.0f}s)", flush=True)
torch.save(model.state_dict(), "/kaggle/working/model.pt"); print("saved")

epoch 1  loss 16.8172  val clause acc 0.751  (19s)
epoch 2  loss 1.5665  val clause acc 0.766  (19s)
epoch 3  loss 0.7405  val clause acc 0.875  (19s)
epoch 4  loss 0.4107  val clause acc 0.875  (19s)
epoch 5  loss 0.2740  val clause acc 0.883  (19s)
epoch 6  loss 0.2024  val clause acc 0.883  (19s)
epoch 7  loss 0.1208  val clause acc 0.927  (19s)
epoch 8  loss 0.1006  val clause acc 0.925  (19s)
saved


In [30]:
print("A: tokenizing", flush=True)
e = tok.encode("The Company shall indemnify the Buyer.")
x = torch.tensor([e.ids]); m = torch.ones_like(x, dtype=torch.bool)
model.eval()
with torch.no_grad():
    print("B: encoder", flush=True)
    h = model.encode(x); print("encoder ok", tuple(h.shape), flush=True)
    print("C: CRF decode", flush=True)
    out = model.crf.decode(model.tag_fc(h), mask=m); print("decode ok", out, flush=True)
print("smoke test passed")

A: tokenizing
B: encoder
encoder ok (1, 7, 256)
C: CRF decode
decode ok [[0, 0, 0, 0, 0, 0, 0]]
smoke test passed


In [31]:
def predict_batch(texts, bs=64):
    model.eval(); res = []
    for i in range(0, len(texts), bs):
        encs = [tok.encode(t) for t in texts[i:i+bs]]
        L = max(max(len(e.ids) for e in encs), 1)
        x = torch.zeros(len(encs), L, dtype=torch.long); m = torch.zeros(len(encs), L, dtype=torch.bool)
        for k, e in enumerate(encs):
            n = len(e.ids)
            if n: x[k, :n] = torch.tensor(e.ids); m[k, :n] = True
        with torch.no_grad():
            tag_ids, logits = model.predict(x, m)
        probs = logits.softmax(-1)
        for k, e in enumerate(encs):
            res.append(([TAGS[j] for j in tag_ids[k]], e.offsets, probs[k]))
    return res

def valid(e):
    t = e["text"]
    if e["label"] in ("MONEY", "DATE"): return any(ch.isdigit() for ch in t)
    if e["label"] == "ORG": return len(t) >= 2 and any(ch.isalpha() for ch in t)
    return True

def spans_from_tags(tags, offs, text):
    ents, cur = [], None
    for t, (a, b) in zip(tags, offs):
        if t == "O":
            if cur: ents.append(cur); cur = None
        elif t.startswith("B-") or cur is None or cur["label"] != t[2:]:
            if cur: ents.append(cur)
            cur = {"start": a, "end": b, "label": t[2:]}
        else:
            cur["end"] = b
    if cur: ents.append(cur)
    out = []
    for e in ents:
        e["text"] = text[e["start"]:e["end"]]
        if valid(e): out.append(e)
    return out

def extract(text, thr=0.9):
    tags, offs, probs = predict_batch([text])[0]
    conf, k = probs.max(0)
    return {"clause_label": CLASSES[int(k)] if float(conf) >= thr else "UNCERTAIN",
            "confidence": round(float(conf), 4), "entities": spans_from_tags(tags, offs, text)}

def split_clauses(doc):
    out = []
    for m in re.finditer(r"[^\n]+", doc):
        base, line, last = m.start(), m.group(), 0
        for s in re.finditer(r"(?<=[.;])\s+(?=[A-Z])", line):
            out.append((base + last, base + s.start())); last = s.end()
        out.append((base + last, base + len(line)))
    return [(a, b) for a, b in out if doc[a:b].strip()]

def process_document(doc, thr=0.9):
    result = []
    for a, b in split_clauses(doc):
        res = extract(doc[a:b], thr)
        for e in res["entities"]:
            e["start"] += a; e["end"] += a
            assert doc[e["start"]:e["end"]] == e["text"]        # zero-hallucination check
        result.append({"clause_span": [a, b], "clause_text": doc[a:b], **res})
    return {"clauses": result}

doc = ("Neither party shall be liable for any delay caused by acts of God, war, flood or pandemic.\n"
       "Acme Corp. shall indemnify and hold harmless Globex Ltd. against all claims arising on 12 March 2025.\n"
       "The total liability of Initech LLC under this Agreement shall not exceed $250,000.\n"
       "This Agreement shall be governed by the laws of India.")
print(json.dumps(process_document(doc), indent=2))
print("\n--- same document with heavy OCR noise ---")
noisy_doc = noise(doc, 0.06, random.Random(5)); print(noisy_doc)
print(json.dumps(process_document(noisy_doc), indent=2))

{
  "clauses": [
    {
      "clause_span": [
        0,
        90
      ],
      "clause_text": "Neither party shall be liable for any delay caused by acts of God, war, flood or pandemic.",
      "clause_label": "Force Majeure",
      "confidence": 0.9969,
      "entities": []
    },
    {
      "clause_span": [
        91,
        192
      ],
      "clause_text": "Acme Corp. shall indemnify and hold harmless Globex Ltd. against all claims arising on 12 March 2025.",
      "clause_label": "Indemnification",
      "confidence": 0.9964,
      "entities": [
        {
          "start": 91,
          "end": 101,
          "label": "ORG",
          "text": "Acme Corp."
        },
        {
          "start": 136,
          "end": 147,
          "label": "ORG",
          "text": "Globex Ltd."
        },
        {
          "start": 178,
          "end": 191,
          "label": "DATE",
          "text": "12 March 2025"
        }
      ]
    },
    {
      "clause_span": [
        193,
    

In [32]:
def clause_preds(preds, thr=0.9):
    out = []
    for _, _, probs in preds:
        conf, k = probs.max(0)
        out.append((CLASSES[int(k)], float(conf)))
    return out

for name, ts in test_sets.items():
    texts = [d["text"] for d in ts]; preds = predict_batch(texts)
    yt = [encode(tok, d["text"], d["spans"])[2] for d in ts]
    yp = [p[0] for p in preds]
    yb = [encode(tok, d["text"], silver_spans(d["text"]))[2] for d in ts]
    print(f"\n===== {name} | unseen company names =====")
    print("-- NER: BiLSTM-CRF --"); pm = ner_scores(yt, yp)
    print("-- NER: regex-only baseline --"); pb = ner_scores(yt, yb)
    print(f"overall NER precision: model {pm:.3f} vs regex {pb:.3f}")
    cp = [p if c >= 0.9 else "UNCERTAIN" for p, c in clause_preds(preds)]
    print("\n-- Clause classification (abstain below 0.9) --")
    print(cls_report([d["clause"] for d in ts], cp, digits=3, zero_division=0))


===== clean | unseen company names =====
-- NER: BiLSTM-CRF --
label    precision   recall       f1
ORG          0.301    0.278    0.289
MONEY        0.992    0.992    0.992
DATE         1.000    0.995    0.997
ALL          0.619    0.591    0.605
-- NER: regex-only baseline --
label    precision   recall       f1
ORG          0.911    0.875    0.893
MONEY        1.000    1.000    1.000
DATE         1.000    1.000    1.000
ALL          0.951    0.930    0.940
overall NER precision: model 0.619 vs regex 0.951

-- Clause classification (abstain below 0.9) --
                         precision    recall  f1-score   support

          Force Majeure      0.963     1.000     0.981        26
        Indemnification      0.871     0.818     0.844        33
Limitation of Liability      0.970     0.552     0.703        58
                  Other      0.988     0.937     0.962       269
              UNCERTAIN      0.000     0.000     0.000         0

               accuracy                     

In [33]:
def sweep(ts, thrs=(0.5, 0.7, 0.8, 0.9, 0.95, 0.99)):
    cp = clause_preds(predict_batch([d["text"] for d in ts]))
    rows = [(d["clause"], p, c) for d, (p, c) in zip(ts, cp)]
    print("thr    risk_precision  risk_recall  abstain_rate")
    for thr in thrs:
        tp = fp = fn = ab = 0
        for true, pred, conf in rows:
            p = pred if conf >= thr else "UNCERTAIN"
            ab += (p == "UNCERTAIN")
            if p in RISKS:
                if p == true: tp += 1
                else: fp += 1
            if true in RISKS and p != true: fn += 1
        print(f"{thr:<6} {tp/max(tp+fp,1):.3f}           {tp/max(tp+fn,1):.3f}        {ab/len(rows):.3f}")

print("== light OCR =="); sweep(test_sets["light OCR"])
print("\n== heavy OCR =="); sweep(test_sets["heavy OCR"])

== light OCR ==
thr    risk_precision  risk_recall  abstain_rate
0.5    0.891           0.838        0.010
0.7    0.894           0.795        0.049
0.8    0.906           0.744        0.073
0.9    0.929           0.675        0.117
0.95   0.974           0.641        0.153
0.99   0.983           0.496        0.269

== heavy OCR ==
thr    risk_precision  risk_recall  abstain_rate
0.5    0.867           0.778        0.005
0.7    0.884           0.718        0.041
0.8    0.906           0.658        0.083
0.9    0.958           0.581        0.137
0.95   0.964           0.462        0.192
0.99   0.974           0.325        0.308


In [34]:
ORG_SUFFIX = re.compile(r"(?:Inc\.?|Incorporated|LLC|L\.L\.C\.|Ltd\.?|Limited|Corp\.?|Corporation|Company|Co\.|LLP|L\.P\.|LP|PLC|plc|GmbH|N\.A\.)$")
def valid_org(t):
    t = t.strip()
    return len(t) >= 4 and " " in t and ORG_SUFFIX.search(t) is not None

def valid(e):                      # overrides the earlier version, used by extract()/process_document()
    t = e["text"]
    if e["label"] in ("MONEY", "DATE"): return any(ch.isdigit() for ch in t)
    if e["label"] == "ORG": return valid_org(t)
    return True

def filter_tags(tags, offs, text):
    tags = list(tags)
    for lab, s, e in get_entities(tags):
        if lab == "ORG" and not valid_org(text[offs[s][0]:offs[e-1][1]]):
            for i in range(s, e): tags[i] = "O"
    return tags

for name, ts in test_sets.items():
    preds = predict_batch([d["text"] for d in ts])
    yt = [encode(tok, d["text"], d["spans"])[2] for d in ts]
    yp = [filter_tags(p[0], p[1], d["text"]) for p, d in zip(preds, ts)]
    yb = [encode(tok, d["text"], silver_spans(d["text"]))[2] for d in ts]
    print(f"\n===== {name} | unseen names | WITH ORG precision filter =====")
    print("-- NER: BiLSTM-CRF + filter --"); pm = ner_scores(yt, yp)
    print("-- NER: regex-only baseline --"); pb = ner_scores(yt, yb)
    print(f"overall NER precision: model {pm:.3f} vs regex {pb:.3f}")


===== clean | unseen names | WITH ORG precision filter =====
-- NER: BiLSTM-CRF + filter --
label    precision   recall       f1
ORG          0.439    0.242    0.312
MONEY        0.992    0.992    0.992
DATE         1.000    0.995    0.997
ALL          0.765    0.571    0.654
-- NER: regex-only baseline --
label    precision   recall       f1
ORG          0.911    0.875    0.893
MONEY        1.000    1.000    1.000
DATE         1.000    1.000    1.000
ALL          0.951    0.930    0.940
overall NER precision: model 0.765 vs regex 0.951

===== light OCR | unseen names | WITH ORG precision filter =====
-- NER: BiLSTM-CRF + filter --
label    precision   recall       f1
MONEY        0.992    0.992    0.992
ORG          0.418    0.213    0.282
DATE         1.000    0.995    0.997
ALL          0.768    0.555    0.644
-- NER: regex-only baseline --
label    precision   recall       f1
MONEY        0.991    0.949    0.970
ORG          0.911    0.774    0.837
DATE         0.988    0.879    0

In [35]:
def process_file(in_path, out_path, thr=0.9):
    with open(in_path, encoding="utf-8", errors="ignore") as f: doc = f.read()
    out = process_document(doc, thr)
    with open(out_path, "w") as f: json.dump(out, f, indent=2)
    return out

sample = "/kaggle/working/sample_contract.txt"
with open(sample, "w") as f: f.write(noisy_doc)           # the OCR-noisy demo document from Cell 8
res = process_file(sample, "/kaggle/working/sample_contract.json")
print(len(res["clauses"]), "clauses written to sample_contract.json")
print(json.dumps(res["clauses"][2], indent=2))             # the noisy liability clause

4 clauses written to sample_contract.json
{
  "clause_span": [
    185,
    262
  ],
  "clause_text": "Thetotal iabilit of Intech LC under this Agreemcnt shall not exceed $250,000.",
  "clause_label": "UNCERTAIN",
  "confidence": 0.6883,
  "entities": [
    {
      "start": 253,
      "end": 261,
      "label": "MONEY",
      "text": "$250,000"
    }
  ]
}


# Legal Clause Classification and Entity Extraction

## Pipeline
OCR text → BPE tokenizer → Word2Vec embeddings (100-d) → BiLSTM → CRF head (NER: ORG/DATE/MONEY) + softmax head (clause class: Indemnification, Limitation of Liability, Force Majeure, Other) → JSON of exact character spans.

## Requirement checklist
| Requirement | How it is met |
|---|---|
| Subword tokenization | BPE (HuggingFace tokenizers), 8k vocab, trained on the corpus |
| Dense embeddings | Word2Vec (skip-gram) over BPE tokens |
| Sequence model | BiLSTM with CRF layer (NER) and pooled softmax (clause class) |
| JSON output with spans | `process_document` returns start/end indices, label, and text for every entity |
| Zero hallucination | Spans are sliced from the source using tokenizer offsets; an assertion checks `doc[start:end] == text` |
| Precision over recall | Clause labels below a confidence threshold become UNCERTAIN; ORG/MONEY/DATE validators drop invalid spans |
| OCR robustness | Training and evaluation use injected OCR noise (character confusions, dropped letters, missing punctuation, merged words) at 3 levels |

## Data
Clause classification: real clauses from the Legal-Clause-Dataset (Kaggle). Leading headings that revealed the label (e.g. "Indemnification.") were stripped (100% of clauses had one). Risk classes: Indemnification (4 source types), Limitation of Liability (2), Force Majeure (1); "Other" = 36 other clause types, capped at 50 each, including look-alikes such as remedies, warranty and set-off.
NER: the dataset has no entity annotations. Labels are **silver**: regex rules for DATE, MONEY and ORG, plus entity injection (defined terms such as "the Company" replaced by generated company names, and neutral sentences with dates/amounts/organisations added). Test uses company names never seen in training.

## Results (held-out test, 386 clauses)
| Test noise | DATE P/R | MONEY P/R | Clause precision on risk classes (threshold 0.9) |
|---|---|---|---|
| Clean | 1.000 / 0.995 | 0.992 / 0.992 | FM 0.963, Indem 0.871, LoL 0.970 |
| Light OCR | 1.000 / 0.995 | 0.992 / 0.992 | FM 0.962, Indem 0.862, LoL 0.967 |
| Heavy OCR | 0.989 / 0.984 | 0.966 / 0.975 | FM 0.955, Indem 0.926, LoL 1.000 |

Under heavy OCR noise the BiLSTM-CRF keeps DATE recall at 0.984 and MONEY recall at 0.975, while the regex-only baseline falls to 0.747 and 0.831.

ORG (before the validator): precision 0.25–0.30 on unseen names; (after validator): see Cell 11 output → ORG P = ___ , R = ___ .

### Precision vs. threshold (clause classes, light OCR)
| Threshold | Risk precision | Risk recall | Abstain rate |
|---|---|---|---|
| 0.5 | 0.891 | 0.838 | 0.010 |
| 0.9 | 0.929 | 0.675 | 0.117 |
| 0.95 | 0.974 | 0.641 | 0.153 |
| 0.99 | 0.983 | 0.496 | 0.269 |
Raising the threshold buys precision at the cost of recall, as required for legal use.

## Limitations
- NER labels are rule-generated (silver), not human-annotated, so NER scores measure agreement with the rules plus injected entities, not true legal-NER accuracy. The regex baseline's clean-text scores are partly circular for the same reason.
- ORG extraction on unseen names is weak (precision ~0.30 before the suffix validator); organisations without a legal suffix are dropped by the validator.
- Limitation of Liability recall is low (0.55 clean, 0.38 heavy OCR); the model abstains instead of guessing.
- Clause splitting is regex-based; some labels in the source data may be noisy.
- Future work: human-annotated NER data, a frozen BERT encoder, a learned clause segmenter.